# 01 · Byte-pair encoding (BPE) tokenizer

**Needed for:** GPT-2 (LLM track, step 1). Every later language model uses a BPE-style tokenizer.

**Idea:** start from raw UTF-8 bytes (256 base tokens). Repeatedly find the most frequent adjacent pair of tokens
and replace it with a new token id. The list of merges, in order, *is* the tokenizer.

- **Training** learns the merges from a corpus.
- **Encoding** replays the merges on new text, earliest-learned merge first.
- **Decoding** maps each id back to its bytes and joins them.

How to use this notebook: run the reference top to bottom, then do the drill at the end without scrolling up.

In [ ]:
import re

def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Counting pairs and merging

In [ ]:
def get_stats(ids, counts=None):
    """Count every adjacent pair. [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}"""
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """Replace every non-overlapping occurrence of `pair` (left to right) with `idx`."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def test_get_stats(fn):
    assert fn([1, 2, 3, 1, 2]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    assert fn([7]) == {}


def test_merge(fn):
    assert fn([1, 2, 3, 1, 2], (1, 2), 99) == [99, 3, 99]
    assert fn([5, 5, 5], (5, 5), 9) == [9, 5]  # non-overlapping, left to right


test_get_stats(get_stats)
test_merge(merge)
print("ok")

## 2. Training, encoding, decoding

`vocab[idx]` holds the bytes for every token, so decoding is just a byte join. Note the `errors="replace"`:
a token sequence can end in the middle of a multi-byte UTF-8 character (for example when a model generates
one token at a time), so decoding a *prefix* must not crash.

In [ ]:
def train_bpe(text, vocab_size):
    assert vocab_size >= 256
    ids = list(text.encode("utf-8"))
    merges = {}                                   # (id, id) -> new id, in the order learned
    vocab = {i: bytes([i]) for i in range(256)}
    for idx in range(256, vocab_size):
        stats = get_stats(ids)
        if not stats:
            break
        pair = max(stats, key=stats.get)
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
    return merges, vocab


def encode(text, merges):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # apply the merge that was learned first (lowest new id)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids


def decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")


corpus = (
    "the cat sat on the mat. the dog sat on the log. "
    "a naïve café in 東京 serves the best coffee. " * 20
)
merges, vocab = train_bpe(corpus, vocab_size=300)

sample = "the cat sat in the café in 東京"
ids = encode(sample, merges)
print("bytes:", len(sample.encode()), "-> tokens:", len(ids))
print("tokens:", [vocab[i] for i in ids])
assert decode(ids, vocab) == sample
print("first merges:", [vocab[i] for i in list(merges.values())[:8]])

## 3. What GPT-2 adds on top

1. **Pre-tokenization with a regex.** Text is split into chunks (words, numbers, punctuation, whitespace)
   and merges never cross chunk boundaries, so you don't get tokens like `"dog."` or `" the ca"`.
   GPT-2 uses the pattern below with the third-party `regex` module (it needs `\p{L}` Unicode classes).
   Here is a simplified version with the standard library:
2. **Special tokens** like `<|endoftext|>` are added outside the merge table and never produced by merges.
3. **Byte-to-unicode mapping** in GPT-2's released code: a cosmetic trick so every byte prints as a character.
   Not needed if you store bytes.

```
GPT-2: 's|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+
```

In [ ]:
SIMPLE_SPLIT = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+")


def train_bpe_chunked(text, vocab_size):
    """Same as train_bpe, but pairs are only counted inside regex chunks."""
    chunks = [list(c.encode("utf-8")) for c in SIMPLE_SPLIT.findall(text)]
    merges, vocab = {}, {i: bytes([i]) for i in range(256)}
    for idx in range(256, vocab_size):
        stats = {}
        for c in chunks:
            get_stats(c, stats)
        if not stats:
            break
        pair = max(stats, key=stats.get)
        chunks = [merge(c, pair, idx) for c in chunks]
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
    return merges, vocab


print(SIMPLE_SPLIT.findall("the dog's log, 42 dogs!"))
m2, v2 = train_bpe_chunked(corpus, 300)
print("chunked merges never contain a '.' next to a letter:",
      not any(re.search(rb"[a-z]\.", v2[i]) for i in m2.values()))

## 4. Using the real GPT-2 tokenizer

For paper implementations, use `tiktoken` (fast, exact GPT-2 / GPT-4 vocabularies) once you understand the above.
`pip install tiktoken` if the cell prints that it is missing.

In [ ]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    toks = enc.encode("the cat sat in the café in 東京")
    print(len(toks), [enc.decode([t]) for t in toks])
    print("vocab size:", enc.n_vocab)  # 50257 = 256 bytes + 50000 merges + <|endoftext|>
except ImportError:
    print("tiktoken not installed; skip")

## Things to be able to explain

- Why start from bytes instead of characters? (No unknown tokens ever; any string is encodable.)
- Why does vocabulary size trade off against sequence length and embedding-table size?
- Why do LLMs struggle with spelling and digit arithmetic? (Tokens hide characters; numbers split unevenly.)
- Why must encoding apply merges in training order rather than "most frequent pair in this text"?

## Drill

Write these cold. Each cell prints PASS once your version matches.

In [ ]:
def my_get_stats(ids):
    raise NotImplementedError


def my_merge(ids, pair, idx):
    raise NotImplementedError


drill(test_get_stats, my_get_stats)
drill(test_merge, my_merge)

In [ ]:
def my_encode(text, merges):
    """Use my_get_stats and my_merge. Must match encode() on any text."""
    raise NotImplementedError


def test_encode(fn):
    for s in ["the cat sat", "café 東京", "", "zzz"]:
        assert fn(s, merges) == encode(s, merges), s


drill(test_encode, my_encode)